In [ ]:
%pip install azure-identity azure-mgmt-costmanagement
dbutils.library.restartPython()

In [ ]:
%run ./utils_common

In [ ]:
import time
import json
import requests
from azure.identity import ClientSecretCredential
from azure.mgmt.costmanagement import CostManagementClient
from azure.mgmt.costmanagement.models import (
    QueryDefinition,
    QueryDataset,
    QueryTimePeriod,
    QueryAggregation,
    QueryGrouping,
    ExportType,
    TimeframeType,
)

In [ ]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")
dbutils.widgets.text("overlap_days", "3")
dbutils.widgets.text("subscription_id", "")
dbutils.widgets.text("scope", "")

In [ ]:
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

In [ ]:
azure_logger = setup_logger("AzureCostExplorer")
logging.getLogger("azure").setLevel(logging.WARNING)
logging.getLogger("azure.core").setLevel(logging.WARNING)
logging.getLogger("azure.identity").setLevel(logging.WARNING)
logging.getLogger("py4j").setLevel(logging.ERROR)

overlap_days = get_overlap_days(dbutils.widgets.get("overlap_days"), logger=azure_logger)

In [ ]:
# Table FQNs are constructed inside AzureCostReporterApp for consistency
# with the dependency-injection pattern used across all notebooks.

In [ ]:
class AzureCostClient:
    def __init__(self, subscription_id, tenant_id, client_id, client_secret, logger=None):
        self.subscription_id = subscription_id
        self.logger = logger or logging.getLogger("AzureCostClient")

        self.credential = ClientSecretCredential(
            tenant_id=tenant_id,
            client_id=client_id,
            client_secret=client_secret
        )

        self.client = CostManagementClient(self.credential)
        self.scope = f"/subscriptions/{self.subscription_id}"
        self.max_chunk_days = 10
        self.max_retries = 3

    # -------- Public API --------
    def group_by_job_clusterid_daily(
        self,
        start_date: datetime,
        end_date: datetime,
        tag_name: str = "clusterid",
    ):
        """Entry point: handles chunking and unions all results.

        Uses dual grouping (TagKey + MeterCategory) to get per-service
        cost breakdown per cluster per day in a single API call.
        """
        start_utc, end_utc = self._to_utc(start_date, end_date)

        chunks = self._build_chunks(start_utc, end_utc, self.max_chunk_days)

        all_chunk_dfs = []
        for chunk_start, chunk_end in chunks:
            self.logger.info(f"Querying chunk {chunk_start} → {chunk_end}")
            df = self._query_with_retries(chunk_start, chunk_end, tag_name)
            if df is not None and df.limit(1).count() > 0:
                all_chunk_dfs.append(df)
            time.sleep(5)

        if not all_chunk_dfs:
            return None

        result_df = all_chunk_dfs[0]
        for df in all_chunk_dfs[1:]:
            result_df = result_df.unionByName(df, allowMissingColumns=True)

        return result_df

    # -------- Helpers: date & chunks --------
    def _to_utc(self, start_date: datetime, end_date: datetime):
        start_utc = start_date.astimezone(timezone.utc)
        end_utc = end_date.astimezone(timezone.utc)
        return start_utc, end_utc

    def _build_chunks(self, start_utc: datetime, end_utc: datetime, max_days: int):
        """Return list of (chunk_start, chunk_end) in UTC."""
        chunks = []
        current = start_utc
        while current <= end_utc:
            chunk_end = min(current + timedelta(days=max_days - 1), end_utc)
            chunks.append((current, chunk_end))
            current = chunk_end + timedelta(days=1)
        return chunks

    # -------- Helpers: query construction --------
    def _build_dataset(self, tag_name: str):
        return QueryDataset(
            granularity="Daily",
            aggregation={"totalCost": QueryAggregation(name="Cost", function="Sum")},
            grouping=[
                QueryGrouping(type="TagKey", name=tag_name),
                QueryGrouping(type="Dimension", name="MeterCategory"),
            ],
        )

    def _build_query_definition(self, start_utc: datetime, end_utc: datetime, dataset):
        return QueryDefinition(
            type=ExportType.ACTUAL_COST,
            timeframe=TimeframeType.CUSTOM,
            time_period=QueryTimePeriod(from_property=start_utc, to=end_utc),
            dataset=dataset,
        )

    def _build_query_body_json(self, start_utc: datetime, end_utc: datetime, tag_name: str):
        body = {
            "type": "ActualCost",
            "timeframe": "Custom",
            "timePeriod": {
                "from": start_utc.isoformat(),
                "to": end_utc.isoformat(),
            },
            "dataset": {
                "granularity": "Daily",
                "aggregation": {
                    "totalCost": {
                        "name": "Cost",
                        "function": "Sum",
                    }
                },
                "grouping": [
                    {"type": "TagKey", "name": tag_name},
                    {"type": "Dimension", "name": "MeterCategory"},
                ],
            },
        }
        return json.dumps(body)

    # -------- Core call with retries --------
    def _query_with_retries(self, start_utc, end_utc, tag_name):
        dataset = self._build_dataset(tag_name)
        query = self._build_query_definition(start_utc, end_utc, dataset)
        query_json = self._build_query_body_json(start_utc, end_utc, tag_name)

        attempt = 0
        last_exception = None

        while attempt < self.max_retries:
            try:
                return self._execute_query(query, query_json)
            except Exception as e:
                last_exception = e
                attempt += 1

                is_429 = "429" in str(e) or "Too many requests" in str(e)
                if attempt >= self.max_retries:
                    break

                if is_429:
                    wait_sec = 30 * attempt
                    self.logger.warning(
                        f"Rate limited (429) on main query, waiting {wait_sec}s (attempt {attempt})..."
                    )
                    time.sleep(wait_sec)
                else:
                    wait_sec = 2 ** attempt
                    self.logger.warning(
                        f"Error on main query, waiting {wait_sec}s (attempt {attempt}): {e}"
                    )
                    time.sleep(wait_sec)

        raise last_exception

    # -------- Single query + pagination --------
    def _execute_query(self, query, query_json: str):
        result = self.client.query.usage(self.scope, parameters=query)

        if not result.rows:
            return None

        col_names = None
        if result.columns:
            col_names = [col.name for col in result.columns]

        all_rows = list(result.rows)

        next_link = getattr(result, "next_link", None)
        token = None
        if next_link:
            token = self.credential.get_token(
                "https://management.azure.com/.default"
            ).token

        while next_link:
            next_link, page_rows = self._fetch_next_page(next_link, token, query_json)
            all_rows.extend(page_rows)
            if next_link:
                time.sleep(2)

        return self._rows_to_df(all_rows, col_names)

    def _fetch_next_page(self, next_link: str, token: str, query_json: str):
        while True:
            resp = requests.post(
                next_link,
                headers={
                    "Authorization": f"Bearer {token}",
                    "Content-Type": "application/json",
                },
                data=query_json,
            )

            if resp.status_code == 429:
                headers = resp.headers
                retry_after = (
                    headers.get("x-ms-ratelimit-microsoft.costmanagement-qpu-retry-after")
                    or headers.get("x-ms-ratelimit-microsoft.costmanagement-entity-retry-after")
                    or headers.get("x-ms-ratelimit-microsoft.costmanagement-tenant-retry-after")
                    or headers.get("x-ms-ratelimit-microsoft.costmanagement-client-retry-after")
                    or headers.get("Retry-After")
                )

                wait_sec = int(retry_after) if retry_after is not None else 30
                self.logger.warning(f"429 throttled, waiting {wait_sec}s before retrying nextLink...")
                time.sleep(wait_sec)
                continue

            resp.raise_for_status()
            data = resp.json()
            props = data.get("properties", {})
            page_rows = props.get("rows", [])
            new_next_link = props.get("nextLink")
            return new_next_link, page_rows

    # -------- Helper: convert rows to DataFrame --------
    def _rows_to_df(self, rows, col_names=None):
        """Build a Spark DataFrame from Azure response rows.

        Requires API-reported column names for deterministic schema mapping.
        Raises SchemaValidationError if column metadata is missing or inconsistent.
        """
        if not col_names or not rows or len(col_names) != len(rows[0]):
            raise SchemaValidationError(
                f"Azure Cost API returned rows with {len(rows[0]) if rows else 0} columns "
                f"but reported {len(col_names) if col_names else 0} column names. "
                f"Cannot construct DataFrame without reliable column metadata."
            )
        df = spark.createDataFrame(rows, col_names)
        for c in df.columns:
            df = df.withColumnRenamed(c, c.lower())
        return df


In [ ]:
# =======================================================
# Azure MeterCategory Classification Framework
# =======================================================
# Single source of truth for Azure MeterCategory classification.
# Category -> list of case-insensitive substring patterns.
# Any MeterCategory not matching a pattern is routed to "other"
# (never silently assigned to compute).
AZURE_METER_CLASSIFICATION = {
    "compute": ["virtual machine"],
    "storage": ["storage", "disk"],
    "network": ["bandwidth", "virtual network", "load balancer", "network watcher"],
}


def classify_azure_meter_category(meter_category: str) -> str:
    """Classify an Azure MeterCategory string using AZURE_METER_CLASSIFICATION.

    Unknown categories return 'other' -- never silently 'compute'.
    """
    if not meter_category:
        return "other"
    lower = meter_category.lower()
    for category, patterns in AZURE_METER_CLASSIFICATION.items():
        for pattern in patterns:
            if pattern in lower:
                return category
    return "other"


def build_azure_category_column(mc_col_name: str):
    """Build a PySpark Column expression from AZURE_METER_CLASSIFICATION.

    Generates the when/otherwise chain programmatically so the dict
    is the sole source of truth. Unknown meters map to 'other'.
    """
    lower_mc = F.lower(F.col(mc_col_name))
    expr = None
    for category, patterns in AZURE_METER_CLASSIFICATION.items():
        cond = lower_mc.contains(patterns[0])
        for p in patterns[1:]:
            cond = cond | lower_mc.contains(p)
        if expr is None:
            expr = F.when(cond, F.lit(category))
        else:
            expr = expr.when(cond, F.lit(category))
    return expr.otherwise(F.lit("other"))


# Known column names the Azure Cost API may use for the tag-value field
_AZURE_TAG_VALUE_CANDIDATES = {"clusterid", "clusteridvalue", "tagvalue"}


def _resolve_cluster_id_column(spark_df, date_col, logger):
    """Rename the Azure tag-value column to ``cluster_id`` if needed.

    Uses only the known candidate column names from _AZURE_TAG_VALUE_CANDIDATES.
    Raises SchemaValidationError if no candidate matches.
    """
    if "cluster_id" in spark_df.columns:
        return spark_df

    tag_value_col = next(
        (c for c in spark_df.columns if c.lower() in _AZURE_TAG_VALUE_CANDIDATES),
        None,
    )

    if tag_value_col is None:
        raise SchemaValidationError(
            f"Cannot resolve cluster_id column from API response. "
            f"Columns present: {spark_df.columns}. "
            f"Expected one of: {sorted(_AZURE_TAG_VALUE_CANDIDATES)}"
        )

    logger.info(f"Resolved cluster_id column from '{tag_value_col}'")
    return spark_df.withColumnRenamed(tag_value_col, "cluster_id")


# =======================================================
# APP
# =======================================================
class AzureCostReporterApp:
    """Orchestrates incremental Azure cost ingestion into the cloud cost table.

    Invariant enforced: cloud_cost = compute_cost + storage_cost + network_cost + other_cost
    """

    TABLE_NAME = "dbspend360_cloud_cost_explorer"

    def __init__(self, catalog, schema, overlap_days, logger):
        self.overlap_days = overlap_days
        self.logger = logger
        self.audit_table = build_table_fqn(catalog, schema, "dbspend360_audit_log")
        self.target_table = build_table_fqn(catalog, schema, "dbspend360_cloud_cost_explorer")
        self.error_log_table = build_table_fqn(catalog, schema, "dbspend360_error_log")
        self.breakdown_table = build_table_fqn(catalog, schema, "dbspend360_other_cost_breakdown")

        scope = dbutils.widgets.get("scope")
        subscription_id = dbutils.widgets.get("subscription_id")
        tenant_id = dbutils.secrets.get(scope, "tenant_id")
        client_id = dbutils.secrets.get(scope, "client_id")
        client_secret = dbutils.secrets.get(scope, "client_secret")

        self.client = AzureCostClient(
            subscription_id, tenant_id, client_id, client_secret,
            logger=self.logger,
        )

    def run(self):
        start_dt = end_dt = datetime.now(timezone.utc).date()
        try:
            start_dt, end_dt = get_date_window(self.audit_table, self.TABLE_NAME, self.overlap_days)

            self.logger.info(
                f"Querying Azure cost from {start_dt} to {end_dt} "
                f"(overlap_days={self.overlap_days})"
            )

            valid, msg = validate_date_window(start_dt, end_dt, self.overlap_days)
            if not valid:
                raise DataQualityError(msg)

            ensure_cost_columns(self.target_table, logger=self.logger)

            spark_df = self.client.group_by_job_clusterid_daily(
                start_date=datetime.combine(start_dt, datetime.min.time(), tzinfo=timezone.utc),
                end_date=datetime.combine(end_dt, datetime.max.time(), tzinfo=timezone.utc),
                tag_name="clusterid",
            )

            quality_msg = f"overlap_days={self.overlap_days}"

            if spark_df is None or spark_df.limit(1).count() == 0:
                self.logger.info("No Azure cost data returned by API for the requested range.")
                merged_row_count = 0
            else:
                date_col = (
                    "usagedate"
                    if "usagedate" in [c.lower() for c in spark_df.columns]
                    else "date_key"
                )
                spark_df = spark_df.withColumn(
                    "cost_incurred_date",
                    F.to_date(F.col(date_col).cast("string"), "yyyyMMdd"),
                )

                spark_df = _resolve_cluster_id_column(spark_df, date_col, self.logger)

                inc_df = filter_valid_cost_rows(spark_df)

                if inc_df.limit(1).count() == 0:
                    self.logger.info(
                        "No incremental rows after filtering by cluster_id and cost_incurred_date."
                    )
                    merged_row_count = 0
                else:
                    # Azure returns the dimension as "MeterCategory" which becomes
                    # "metercategory" after _rows_to_df lowercases columns. Accept both
                    # spellings so the check is resilient to API/SDK changes.
                    _METER_COL_CANDIDATES = {"metercategory", "meter_category"}
                    mc_col = next(
                        (c for c in inc_df.columns if c.lower() in _METER_COL_CANDIDATES),
                        None,
                    )
                    has_meter = mc_col is not None
                    if has_meter:
                        self._log_unclassified_meters(inc_df, mc_col)

                        classified = inc_df.withColumn("category", build_azure_category_column(mc_col))
                        write_other_cost_breakdown(
                            classified, mc_col, "AZURE", self.breakdown_table, logger=self.logger,
                        )

                        agg_df = aggregate_costs_by_category(classified)
                    else:
                        self.logger.warning(
                            "MeterCategory column not found; all cost assigned to cloud_cost only."
                        )
                        agg_df = (
                            inc_df
                            .groupBy("cluster_id", "currency", "cost_incurred_date")
                            .agg(F.sum("cost").alias("cloud_cost"))
                            .withColumn("compute_cost", F.lit(None).cast("double"))
                            .withColumn("storage_cost", F.lit(None).cast("double"))
                            .withColumn("network_cost", F.lit(None).cast("double"))
                            .withColumn("other_cost", F.lit(None).cast("double"))
                            .withColumn("created_at", F.current_timestamp())
                            .withColumn("updated_at", F.current_timestamp())
                        )

                    agg_df = safe_cache(agg_df)
                    merged_row_count = agg_df.count()

                    validate_source_schema(
                        agg_df,
                        {"cluster_id": "string", "currency": "string",
                         "cost_incurred_date": "date", "cloud_cost": "double"},
                        self.target_table, self.logger,
                    )
                    validate_no_negative_costs(
                        agg_df,
                        ["cloud_cost", "compute_cost", "storage_cost", "network_cost", "other_cost"],
                        self.target_table, self.logger,
                    )
                    validate_currency_consistency(agg_df, "currency", self.target_table, self.logger)

                    quality_msg = compute_quality_metrics(
                        agg_df, merged_row_count, self.overlap_days, logger=self.logger,
                    )

                    merge_cloud_cost_explorer(self.target_table, agg_df)
                    safe_unpersist(agg_df)

                    merge_metrics = get_merge_metrics(self.target_table, self.logger)
                    quality_msg += (
                        f", merge_inserted={merge_metrics.get('num_inserted', '?')}"
                        f", merge_updated={merge_metrics.get('num_updated', '?')}"
                    )

                    validate_post_merge(
                        self.target_table, "cost_incurred_date",
                        start_dt, end_dt, merged_row_count, self.logger,
                    )

            self.logger.info(
                f"Merged {merged_row_count} rows into {self.target_table} "
                f"for {start_dt} → {end_dt} (overlap_days={self.overlap_days})."
            )

            log_audit_run(
                self.audit_table, self.TABLE_NAME, start_dt, end_dt,
                "SUCCESS", merged_row_count, quality_msg,
            )

        except Exception as e:
            msg = str(e)[:1000]
            self.logger.error(f"Run failed: {msg}")
            try:
                log_audit_run(
                    self.audit_table, self.TABLE_NAME, start_dt, end_dt, "FAILED", 0, msg,
                )
            except Exception:
                self.logger.error("Failed to write FAILED audit entry")
            raise

    def _log_unclassified_meters(self, df, mc_col):
        """Log unclassified MeterCategory values to both logger and error_log table."""
        meter_costs = (
            df.groupBy(mc_col)
            .agg(
                F.sum("cost").alias("total_cost"),
                F.count("*").alias("row_count"),
            )
            .collect()
        )
        unknown_rows = [
            r for r in meter_costs
            if r[0] and classify_azure_meter_category(r[0]) == "other"
        ]

        if not unknown_rows:
            return

        meter_names = [r[0] for r in unknown_rows]
        self.logger.warning(
            f"Unclassified Azure MeterCategory values (routed to other_cost): {meter_names}"
        )

        try:
            error_details = [
                f"Unclassified meter: {r[0]}, total_cost=${r.total_cost:.4f}, rows={r.row_count}"
                for r in unknown_rows
            ]
            write_error_log_entries(error_details, "AZURE", "UNCLASSIFIED_COST", self.error_log_table)
        except Exception as e:
            self.logger.warning(f"Failed to write unclassified meters to error_log: {e}")

In [ ]:
# =======================================================
# Execute
# =======================================================
app = AzureCostReporterApp(catalog, schema, overlap_days, azure_logger)
app.run()